In [1]:
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone, ServerlessSpec
from pathlib import Path
from bs4 import BeautifulSoup
import os
from dotenv import load_dotenv

load_dotenv()

gemini_api_key = os.getenv("GEMINI_API_KEY")

pinecone_api_key = os.getenv("PINECONE_KEY")

c:\Users\kshit\OneDrive\Desktop\CodingNinjasAICourse\Rag\Vector Databases\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
HTML_DIR = Path(r'coffee_pages')

In [3]:
HTML_DIR

WindowsPath('coffee_pages')

In [8]:
def get_html_text(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        soup = BeautifulSoup(file, 'html.parser')
        text = soup.get_text(separator=" ", strip=True)
    return text

In [14]:
docs = []

In [15]:
for file_path in HTML_DIR.glob('*.html'):
    text = get_html_text(file_path)
    print(f"Extracted text from {file_path.name}:")
    docs.append(text)

print(f"Total documents extracted: {len(docs)}")

Extracted text from 01_ashwagandha_coffee_adaptogenic_latte.html:
Extracted text from 02_masala_coffee_spiced_indian_coffee.html:
Extracted text from 03_turmeric_coffee_haldi_cappuccino.html:
Extracted text from 04_south_indian_filter_coffee_with_chicory.html:
Extracted text from 05_beaten_coffee_phenti_hui_coffee.html:
Extracted text from 06_cardamom_coffee_elaichi_coffee.html:
Extracted text from 07_ginger_coffee_adrak_coffee.html:
Extracted text from 08_tulsi_holy_basil_coffee.html:
Extracted text from 09_jaggery_coffee_gur_coffee.html:
Extracted text from 10_saffron_coffee_kesar_coffee.html:
Extracted text from 11_pepper_coffee_kali_mirch_coffee.html:
Extracted text from 12_coconut_milk_coffee_south_coastal_style.html:
Extracted text from 13_mushroom_and_ashwagandha_coffee_adaptogen_blend.html:
Extracted text from 14_cold_brew_with_indian_spices.html:
Extracted text from 15_ayurvedic_lens_on_coffee_consumption.html:
Total documents extracted: 15


In [ ]:
INDEX_NAME = "vector-db-coffee"

In [16]:
pinecone = Pinecone(api_key=pinecone_api_key)

In [ ]:
existing_indexes = [idx["name"] for idx in pinecone.list_indexes()]

if INDEX_NAME not in existing_indexes:
    index = pinecone.create_index(
        name=INDEX_NAME,
        dimension=384,          # 384-dim for MiniLM
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1",
        ),
    )
    print(f"✅ Created index: {INDEX_NAME}")
else:
    print(f"ℹ️ Index already exists: {INDEX_NAME}")

ℹ️ Index already exists: coffee-pages-index


In [30]:
pinecone_index = pinecone.Index(INDEX_NAME)

In [31]:
model = SentenceTransformer("all-MiniLM-L6-v2")

In [36]:
get_organized_docs = []

for idx, filepath in enumerate(HTML_DIR.glob("*.html")):

    text_content = get_html_text(filepath)

    get_organized_docs.append({"id":idx, "text":text_content, "location":filepath})

In [37]:
def make_embeddings(model, text_list):
    embeddings = model.encode(text_list, convert_to_numpy=True)
    return embeddings

In [38]:
embeddings_vec = []

for text_details in get_organized_docs:

    vec = make_embeddings(model, text_details['text'])

    embeddings_vec.append({'id': str(text_details['id']), "values": vec, 'metadata':{"text":text_details['text'], "location":text_details['location'].name}})

In [39]:
pinecone_index.upsert(embeddings_vec)

UpsertResponse(upserted_count=15, _response_info={'raw_headers': {'date': 'Sat, 13 Dec 2025 08:07:56 GMT', 'content-type': 'application/json', 'content-length': '20', 'connection': 'keep-alive', 'x-pinecone-request-lsn': '1', 'x-pinecone-request-logical-size': '54625', 'x-pinecone-request-latency-ms': '1409', 'x-pinecone-request-id': '6360634914604838765', 'x-envoy-upstream-service-time': '321', 'grpc-status': '0', 'server': 'envoy'}})